# Data Analysis, Visualisation and Preprocessing

This notebook loads and analyses the three raw data files used in the COS30019 Assignment 2B project:

1. `Scats Data October 2006.xls`
2. `SCATSSiteListingSpreadsheet_VicRoads.xls`
3. `Traffic_Count_Locations_with_LONG_LAT.csv`

Main goals:

- Load the raw files.
- Explore data quality.
- Visualise useful patterns.
- Decide which files need preprocessing before being used in the project.


## 1. Setup

In [3]:
from pathlib import Path
import sys
import subprocess
import os
import re

# Install missing packages for this notebook kernel
required_packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "xlrd"
]

for package in required_packages:
    try:
        __import__(package)
    except ModuleNotFoundError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

print("Current working directory:", BASE_DIR)
print("Processed output directory:", PROCESSED_DIR)

Installing pandas...
Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


Installing matplotlib...
Defaulting to user installation because normal site-packages is not writeable
Installing xlrd...


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


Current working directory: /Users/truonghuuminhhai/Documents/COS30019/Assignment2B/cos30019_assignment2_b/data
Processed output directory: /Users/truonghuuminhhai/Documents/COS30019/Assignment2B/cos30019_assignment2_b/data/processed


In [ ]:
def find_file(possible_names):
    # Search common folders relative to this notebook.
    search_dirs = [
        BASE_DIR,
        BASE_DIR / "raws",
        BASE_DIR / "raw",
        BASE_DIR.parent,
        BASE_DIR.parent / "data",
        BASE_DIR.parent / "data" / "raws",
        BASE_DIR.parent / "data" / "raw",
    ]

    for folder in search_dirs:
        for name in possible_names:
            candidate = folder / name
            if candidate.exists():
                return candidate

    raise FileNotFoundError(f"Could not find any of these files: {possible_names}")

traffic_file = find_file([
    "Scats Data October 2006.xls",
    "SCATS Data October 2006.xls",
])

site_listing_file = find_file([
    "SCATSSiteListingSpreadsheet_VicRoads.xls",
    "SCATSSiteListingSpreadsheet_VicRoads(1).xls",
])

locations_file = find_file([
    "Traffic_Count_Locations_with_LONG_LAT.csv",
    "Traffic_Count_Locations_with_LONG_LAT(1).csv",
])

print("Traffic data file:", traffic_file)
print("SCATS site listing file:", site_listing_file)
print("Traffic count locations file:", locations_file)


# 2. SCATS Traffic Flow Data

File:

```text
Scats Data October 2006.xls
```

This is the main dataset for model training. It contains 96 traffic readings per day, where each reading represents a 15-minute interval.


In [ ]:
# Load the main traffic data.
# Header is on row index 1 in the Excel sheet.
traffic_raw = pd.read_excel(traffic_file, sheet_name="Data", header=1)

print("Traffic raw shape:", traffic_raw.shape)
display(traffic_raw.head())
display(traffic_raw.info())


In [ ]:
# Basic data quality check
print("Columns:", traffic_raw.columns.tolist()[:15], "...")

v_cols = [c for c in traffic_raw.columns if isinstance(c, str) and re.fullmatch(r"V\d{2}", c)]
print("Number of 15-minute interval columns:", len(v_cols))
print("First interval columns:", v_cols[:5])
print("Last interval columns:", v_cols[-5:])

print("\nUnique SCATS sites:", traffic_raw["SCATS Number"].nunique())
print("Date range:", traffic_raw["Date"].min(), "to", traffic_raw["Date"].max())
print("Rows:", len(traffic_raw))

print("\nMissing values in key columns:")
display(traffic_raw[["SCATS Number", "Location", "NB_LATITUDE", "NB_LONGITUDE", "Date"]].isna().sum())

print("\nMissing values in traffic interval columns:")
display(traffic_raw[v_cols].isna().sum().sort_values(ascending=False).head(10))

print("\nTraffic value summary:")
display(traffic_raw[v_cols].describe().T[["min", "mean", "max"]].head())


In [ ]:
# Convert wide format to long time-series format.
traffic_long = traffic_raw.melt(
    id_vars=["SCATS Number", "Location", "NB_LATITUDE", "NB_LONGITUDE", "Date"],
    value_vars=v_cols,
    var_name="interval_code",
    value_name="flow"
)

traffic_long["site_id"] = traffic_long["SCATS Number"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(4)
traffic_long["date_only"] = pd.to_datetime(traffic_long["Date"]).dt.date
traffic_long["interval"] = traffic_long["interval_code"].str.extract(r"V(\d{2})").astype(int)
traffic_long["time"] = pd.to_timedelta(traffic_long["interval"] * 15, unit="m")
traffic_long["timestamp"] = pd.to_datetime(traffic_long["date_only"].astype(str)) + traffic_long["time"]

# Aggregate by site and timestamp.
# Some sites may have multiple detector rows/directions, so we sum flow at each site and interval.
traffic_ts = (
    traffic_long
    .groupby(["site_id", "Location", "NB_LATITUDE", "NB_LONGITUDE", "date_only", "interval", "timestamp"], as_index=False)["flow"]
    .sum()
    .sort_values(["site_id", "timestamp"])
)

print("Long raw shape:", traffic_long.shape)
print("Aggregated time-series shape:", traffic_ts.shape)
display(traffic_ts.head())


In [ ]:
# Data mining summary for traffic flow
print("Number of sites:", traffic_ts["site_id"].nunique())
print("Number of dates:", traffic_ts["date_only"].nunique())
print("Number of intervals per day:", traffic_ts["interval"].nunique())
print("Total zero flow records:", (traffic_ts["flow"] == 0).sum())
print("Missing flow records:", traffic_ts["flow"].isna().sum())

site_summary = traffic_ts.groupby("site_id")["flow"].agg(["count", "mean", "min", "max", "std"]).reset_index()
display(site_summary.sort_values("mean", ascending=False).head(10))

daily_summary = traffic_ts.groupby("date_only")["flow"].agg(["mean", "sum", "max"]).reset_index()
display(daily_summary.head())


## 2.1 Traffic Flow Visualisation

In [ ]:
# Average traffic flow by time of day
avg_by_interval = traffic_ts.groupby("interval")["flow"].mean().reset_index()
avg_by_interval["time_label"] = (pd.to_datetime("2000-01-01") + pd.to_timedelta(avg_by_interval["interval"] * 15, unit="m")).dt.strftime("%H:%M")

plt.figure(figsize=(14, 5))
plt.plot(avg_by_interval["interval"], avg_by_interval["flow"], marker="o")
plt.xticks(avg_by_interval["interval"][::8], avg_by_interval["time_label"][::8], rotation=45)
plt.title("Average Traffic Flow by Time of Day")
plt.xlabel("Time of Day")
plt.ylabel("Average Flow")
plt.tight_layout()
plt.show()


In [ ]:
# Traffic distribution
plt.figure(figsize=(10, 5))
plt.hist(traffic_ts["flow"].dropna(), bins=50)
plt.title("Distribution of Aggregated Traffic Flow")
plt.xlabel("Flow")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Top 10 sites by average traffic
top_sites = site_summary.sort_values("mean", ascending=False).head(10)

plt.figure(figsize=(12, 5))
plt.bar(top_sites["site_id"], top_sites["mean"])
plt.title("Top 10 SCATS Sites by Average Traffic Flow")
plt.xlabel("SCATS Site")
plt.ylabel("Average Flow")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Example one-site time series
example_site = site_summary.sort_values("mean", ascending=False)["site_id"].iloc[0]
example_data = traffic_ts[traffic_ts["site_id"] == example_site].sort_values("timestamp")

plt.figure(figsize=(14, 5))
plt.plot(example_data["timestamp"], example_data["flow"])
plt.title(f"Traffic Flow Over Time for SCATS Site {example_site}")
plt.xlabel("Timestamp")
plt.ylabel("Flow")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Save processed traffic time-series.
traffic_ts.to_csv(PROCESSED_DIR / "traffic_timeseries.csv", index=False)

# Save coordinates extracted from the traffic file.
traffic_coords = (
    traffic_ts[["site_id", "Location", "NB_LATITUDE", "NB_LONGITUDE"]]
    .drop_duplicates("site_id")
    .rename(columns={
        "Location": "location",
        "NB_LATITUDE": "latitude",
        "NB_LONGITUDE": "longitude",
    })
    .sort_values("site_id")
)

traffic_coords.to_csv(PROCESSED_DIR / "scats_coordinates_from_traffic.csv", index=False)

print("Saved:", PROCESSED_DIR / "traffic_timeseries.csv")
print("Saved:", PROCESSED_DIR / "scats_coordinates_from_traffic.csv")
display(traffic_coords.head())


# 3. SCATS Site Listing Data

File:

```text
SCATSSiteListingSpreadsheet_VicRoads.xls
```

This file is metadata. It is useful for mapping SCATS site numbers to intersection names.


In [ ]:
def load_site_listing(path):
    # The real header row is usually row 9 in the 'SCATS Site Numbers' sheet.
    # If this fails because of old XLS formatting, open the file in Excel/LibreOffice and save it as .xlsx, then rerun.
    site_df = pd.read_excel(path, sheet_name="SCATS Site Numbers", header=9)
    site_df = site_df.dropna(how="all")
    site_df.columns = (
        site_df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    if "site_number" in site_df.columns:
        site_df["site_id"] = (
            site_df["site_number"]
            .astype(str)
            .str.replace(r"\.0$", "", regex=True)
            .str.strip()
            .str.zfill(4)
        )

    return site_df

site_listing = load_site_listing(site_listing_file)

print("Site listing shape:", site_listing.shape)
display(site_listing.head())
display(site_listing.info())


In [ ]:
# Site listing data quality
print("Columns:", site_listing.columns.tolist())

if "site_id" in site_listing.columns:
    print("Unique site IDs:", site_listing["site_id"].nunique())
    print("Duplicated site IDs:", site_listing["site_id"].duplicated().sum())
    display(site_listing[site_listing["site_id"].duplicated(keep=False)].head(10))

print("\nMissing values:")
display(site_listing.isna().sum())

# Check how many traffic sites appear in site listing.
traffic_site_ids = set(traffic_ts["site_id"].unique())
listing_site_ids = set(site_listing["site_id"].dropna().unique())

matched_sites = traffic_site_ids.intersection(listing_site_ids)
missing_in_listing = sorted(traffic_site_ids - listing_site_ids)

print("Traffic sites:", len(traffic_site_ids))
print("Matched in site listing:", len(matched_sites))
print("Missing from site listing:", missing_in_listing[:20])


## 3.1 Site Listing Visualisation

In [ ]:
# Site type counts
if "site_type" in site_listing.columns:
    site_type_counts = site_listing["site_type"].value_counts(dropna=False)
    display(site_type_counts.head(10))

    plt.figure(figsize=(8, 5))
    site_type_counts.head(10).plot(kind="bar")
    plt.title("SCATS Site Listing: Site Type Counts")
    plt.xlabel("Site Type")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
# Save cleaned site listing.
site_listing_clean = site_listing.copy()

keep_cols = [c for c in ["site_id", "site_number", "location_description", "site_type", "directory", "map_reference"] if c in site_listing_clean.columns]
site_listing_clean = site_listing_clean[keep_cols].drop_duplicates()

site_listing_clean.to_csv(PROCESSED_DIR / "scats_site_listing_clean.csv", index=False)

print("Saved:", PROCESSED_DIR / "scats_site_listing_clean.csv")
display(site_listing_clean.head())


# 4. Traffic Count Locations Data

File:

```text
Traffic_Count_Locations_with_LONG_LAT.csv
```

This file contains longitude/latitude and traffic count location metadata. It can be used for map exploration, distance checking, or optional route visualisation.

Important: do not assume `TFM_ID` always equals `SCATS Number` without checking.


In [ ]:
locations_raw = pd.read_csv(locations_file)

print("Locations raw shape:", locations_raw.shape)
display(locations_raw.head())
display(locations_raw.info())


In [ ]:
# Clean column names
locations = locations_raw.copy()
locations.columns = (
    locations.columns
    .astype(str)
    .str.strip()
    .str.lower()
)

rename_map = {
    "x": "longitude",
    "y": "latitude",
    "tfm_id": "tfm_id",
    "tfm_desc": "tfm_desc",
    "site_desc": "site_desc",
    "aadt_allve": "aadt_all_vehicles",
    "aadt_truck": "aadt_truck",
    "per_trucks": "percent_trucks",
}
locations = locations.rename(columns=rename_map)

if "tfm_id" in locations.columns:
    locations["tfm_id"] = locations["tfm_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

print("Cleaned columns:", locations.columns.tolist())
display(locations.head())


In [ ]:
# Location data quality checks
print("Rows:", len(locations))
print("Columns:", len(locations.columns))

for col in ["tfm_id", "longitude", "latitude"]:
    if col in locations.columns:
        print(f"{col}: missing =", locations[col].isna().sum())

if "tfm_id" in locations.columns:
    print("Unique TFM IDs:", locations["tfm_id"].nunique())
    print("Duplicated TFM IDs:", locations["tfm_id"].duplicated().sum())

if {"longitude", "latitude"}.issubset(locations.columns):
    print("\nCoordinate summary:")
    display(locations[["longitude", "latitude"]].describe())

if "aadt_all_vehicles" in locations.columns:
    print("\nAADT summary:")
    display(locations["aadt_all_vehicles"].describe())

if "tfm_id" in locations.columns:
    location_ids = set(locations["tfm_id"].dropna().unique())
    direct_matches = traffic_site_ids.intersection(location_ids)
    not_matched = sorted(traffic_site_ids - location_ids)

    print("Traffic SCATS sites:", len(traffic_site_ids))
    print("Direct matches with TFM_ID:", len(direct_matches))
    print("Traffic SCATS sites not directly matched in TFM_ID:", not_matched)


## 4.1 Traffic Count Location Visualisation

In [ ]:
# Scatter plot of all traffic count locations
if {"longitude", "latitude"}.issubset(locations.columns):
    plot_locations = locations.dropna(subset=["longitude", "latitude"])

    plt.figure(figsize=(8, 8))
    plt.scatter(plot_locations["longitude"], plot_locations["latitude"], s=5, alpha=0.4)
    plt.title("Traffic Count Locations: Longitude/Latitude Scatter Plot")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.show()


In [ ]:
# Overlay SCATS coordinates extracted from traffic data
if {"longitude", "latitude"}.issubset(locations.columns):
    valid_scats_coords = traffic_coords.copy()
    valid_scats_coords = valid_scats_coords[
        (valid_scats_coords["latitude"].notna()) &
        (valid_scats_coords["longitude"].notna()) &
        ~((valid_scats_coords["latitude"] == 0) & (valid_scats_coords["longitude"] == 0))
    ]

    plt.figure(figsize=(8, 8))
    plt.scatter(plot_locations["longitude"], plot_locations["latitude"], s=5, alpha=0.25, label="Traffic count locations")
    plt.scatter(valid_scats_coords["longitude"], valid_scats_coords["latitude"], s=40, label="SCATS sites from traffic file")
    plt.title("Traffic Count Locations vs SCATS Coordinates from Traffic Data")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("SCATS coordinates with latitude/longitude = 0:")
    display(traffic_coords[(traffic_coords["latitude"] == 0) | (traffic_coords["longitude"] == 0)])


In [ ]:
# AADT distribution
if "aadt_all_vehicles" in locations.columns:
    plt.figure(figsize=(10, 5))
    plt.hist(locations["aadt_all_vehicles"].dropna(), bins=50)
    plt.title("Distribution of AADT All Vehicles")
    plt.xlabel("AADT All Vehicles")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


In [ ]:
# Save cleaned locations data.
location_keep_cols = [
    c for c in [
        "tfm_id", "tfm_desc", "site_desc", "longitude", "latitude",
        "aadt_all_vehicles", "aadt_truck", "percent_trucks",
        "tfm_typ_de", "movement_t"
    ]
    if c in locations.columns
]

locations_clean = locations[location_keep_cols].drop_duplicates()
locations_clean.to_csv(PROCESSED_DIR / "traffic_count_locations_clean.csv", index=False)

print("Saved:", PROCESSED_DIR / "traffic_count_locations_clean.csv")
display(locations_clean.head())


# 5. Final Decision: Which Files Need Processing?

## `Scats Data October 2006.xls`

Needs heavy preprocessing before model training:

- Convert wide format `V00`–`V95` into time-series format.
- Aggregate detector readings by SCATS site and time interval.
- Scale traffic flow values.
- Create sequences for LSTM/GRU/CNN.
- Split train/test by time.

## `SCATSSiteListingSpreadsheet_VicRoads.xls`

Needs light preprocessing:

- Skip intro rows.
- Use correct header row.
- Standardise column names.
- Standardise site IDs.
- Remove duplicates if needed.
- Save clean lookup table for intersection names.

## `Traffic_Count_Locations_with_LONG_LAT.csv`

Needs light preprocessing if used:

- Standardise column names.
- Validate longitude/latitude.
- Check duplicates.
- Do not directly assume `TFM_ID = SCATS Number`.
- Use mostly for map/data exploration unless mapping is verified.

## Recommended project usage

- Train ML models mainly from `traffic_timeseries.csv` or the arrays generated from it.
- Use `scats_site_listing_clean.csv` for intersection names.
- Use `scats_coordinates_from_traffic.csv` as the primary coordinate source for the 40 SCATS sites in the traffic dataset.
- Use `traffic_count_locations_clean.csv` only as supplementary location/reference data unless the ID mapping is verified.
